# 07 Amount-aware decisions

Each transaction gets the action with the lowest expected cost, based on its fraud probability and amount.

**Findings**
- Calibration holds across score ranges (predicted vs actual: 0.0022 vs 0.0031, 0.1598 vs 0.1410, 0.9902 vs 0.9919), so scores can be used as real probabilities.
- Expected-cost rule vs tuned thresholds (validation):
  - Total cost: $21,076 vs $22,991.
  - Fraud loss: $10,502 vs $12,387.
  - Genuine customers asked for an OTP or blocked: 1,619 vs 2,348.
  - Genuine purchases under $30 blocked: 4 vs 57.
- The rule approves more frauds by count (67 vs 24), but they are small (median $16.41), so less money is lost.
- It beat thresholds retuned for each scenario under all four cost assumptions, with no tuning of its own.
- Weak spot: it blocks some large genuine purchases (114 at a median of $1,086.32) to stop 14 frauds.
- Decision recorded in `models/policy.json`: `decision_rule` = `expected_cost`.

In [1]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")
import json, numpy as np, pandas as pd, joblib
from src.data import load_train_val
from src.features import FEATURES
from src.policy import decide_thresholds, decide_expected_cost, realized_cost

pd.set_option("display.width", 200)
tr, val = load_train_val("../data/raw/fraudTrain.csv")
model = joblib.load("../models/xgb_v1.joblib")
policy = json.load(open("../models/policy.json"))
C = policy["costs"]

val["score"] = model.predict_proba(val[FEATURES])[:, 1]
y, amt, p = val["is_fraud"].to_numpy(), val["amt"].to_numpy(), val["score"].to_numpy()
labels = np.where(y == 1, "fraud", "legit")
no_model = amt[y == 1].sum()

In [2]:
bins = [0, 0.001, 0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 0.9, 1.0]
val["bin"] = pd.cut(val["score"], bins, include_lowest=True)
calib = val.groupby("bin", observed=True).agg(n=("is_fraud", "size"),
                                              mean_score=("score", "mean"),
                                              fraud_rate=("is_fraud", "mean"))
print(calib.round(4).to_string())

                      n  mean_score  fraud_rate
bin                                            
(-0.001, 0.001]  361656      0.0000      0.0000
(0.001, 0.005]     4901      0.0022      0.0031
(0.005, 0.01]      1032      0.0071      0.0058
(0.01, 0.05]       1248      0.0225      0.0337
(0.05, 0.1]         319      0.0702      0.0846
(0.1, 0.25]         312      0.1598      0.1410
(0.25, 0.5]         199      0.3593      0.4171
(0.5, 0.9]          297      0.7372      0.7306
(0.9, 1.0]         1861      0.9902      0.9919


In [3]:
policies = {
    "tuned thresholds": decide_thresholds(p, policy["t_otp"], policy["t_block"]),
    "expected cost":    decide_expected_cost(p, amt, C),
}
for name, actions in policies.items():
    loss, friction = realized_cost(actions, y, amt, C)
    print(f"{name:16s} | fraud loss ${loss:>9,.0f} | friction ${friction:>7,.0f} | "
          f"total ${loss + friction:>9,.0f} | cost cut {1 - (loss + friction) / no_model:.2%}")
    print(pd.crosstab(labels, actions, rownames=["actual"], colnames=["action"]), "\n")

tuned thresholds | fraud loss $   12,387 | friction $ 10,604 | total $   22,991 | cost cut 98.12%
action  approve  block   otp
actual                      
fraud        24   2146   116
legit    367191    211  2137 

expected cost    | fraud loss $   10,502 | friction $ 10,574 | total $   21,076 | cost cut 98.27%
action  approve  block   otp
actual                      
fraud        67   2056   163
legit    367920    262  1357 



In [4]:
for name, a in policies.items():
    n = ((y == 0) & (a == "block") & (amt < C["block_friction"])).sum()
    print(f"{name:16s} | genuine customers blocked on purchases under ${C['block_friction']:.0f}: {n}")

diff = pd.DataFrame({"thresholds": policies["tuned thresholds"],
                     "expected_cost": policies["expected cost"],
                     "label": labels, "amt": amt})
diff = diff[diff["thresholds"] != diff["expected_cost"]]
print(diff.groupby(["thresholds", "expected_cost", "label"])["amt"]
          .agg(count="size", median_amt="median").round(2).to_string())

tuned thresholds | genuine customers blocked on purchases under $30: 57
expected cost    | genuine customers blocked on purchases under $30: 4
                                count  median_amt
thresholds expected_cost label                   
approve    otp           fraud      1      950.46
                         legit    394     1009.16
block      otp           fraud    104       17.31
                         legit     63       19.33
otp        approve       fraud     44       16.41
                         legit   1123       16.87
           block         fraud     14      919.92
                         legit    114     1086.32


In [6]:
scenarios = {
    "base":             C,
    "expensive blocks": {**C, "block_friction": 100.0},
    "annoying OTP":     {**C, "otp_friction": 5.0},
    "weak OTP":         {**C, "otp_stop_rate": 0.5},
}
for name, c in scenarios.items():
    loss, friction = realized_cost(decide_expected_cost(p, amt, c), y, amt, c)
    print(f"{name:16s} | total ${loss + friction:>9,.0f} | cost cut {1 - (loss + friction) / no_model:.2%}")

base             | total $   21,076 | cost cut 98.27%
expensive blocks | total $   30,875 | cost cut 97.47%
annoying OTP     | total $   23,880 | cost cut 98.04%
weak OTP         | total $   25,789 | cost cut 97.89%


In [7]:
policy["decision_rule"] = "expected_cost"
with open("../models/policy.json", "w") as f:
    json.dump(policy, f, indent=2)
print(json.dumps(policy, indent=2))

{
  "t_otp": 0.0075,
  "t_block": 0.25,
  "costs": {
    "otp_friction": 2.0,
    "block_friction": 30.0,
    "otp_stop_rate": 0.8
  },
  "decision_rule": "expected_cost"
}
